# Práctica 4: Clustering — Online Shoppers Intention

**Autor:** Javier Dibo
**Asignatura:** Inteligencia de Negocio

## Objetivos

El objetivo de esta práctica es afrontar un problema de **clustering** sobre un conjunto de datos reales de comercio electrónico, aplicando distintos tipos de algoritmos (particional, jerárquico, basado en densidad y basado en modelos), explorando la configuración de cada uno, y comparando sus resultados para evaluar cuál resulta más adecuado para la segmentación de usuarios por intención de compra.


## Parte I: Descripción del problema

Se dispone del fichero `online_shoppers_intention.xlsx`, que contiene información de **12.330 sesiones de usuario** sobre una web de comercio electrónico, recogidas a lo largo de un año para evitar sesgos estacionales. Cada sesión se describe mediante **18 variables** numéricas y categóricas.

El objetivo es **descubrir grupos (clústeres) de usuarios con comportamientos similares** que permitan segmentar la base de visitantes y, potencialmente, dirigir acciones de marketing personalizadas hacia aquellos con mayor intención de compra.

La variable `Revenue` indica si la sesión finalizó con una transacción (15,5% de las sesiones). **Esta variable se excluye del proceso de clustering** y se reserva exclusivamente para evaluar, *a posteriori*, la calidad de los clústeres obtenidos como estrategia de segmentación.

### Atributos

**Numéricas:** `Administrative`, `Administrative_Duration`, `Informational`, `Informational_Duration`, `ProductRelated`, `ProductRelated_Duration`, `BounceRates`, `ExitRates`, `PageValues`, `SpecialDay`.

**Categóricas:** `Month`, `OperatingSystems`, `Browser`, `Region`, `TrafficType`, `VisitorType`, `Weekend`, `Revenue`.


## Parte II: Carga de datos y análisis exploratorio

### 1. Importación de librerías


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans, AgglomerativeClustering, DBSCAN
from sklearn.mixture import GaussianMixture
from sklearn.metrics import silhouette_score
from sklearn.neighbors import NearestNeighbors

from scipy.stats import entropy
from scipy.cluster.hierarchy import dendrogram, linkage

sns.set_style('whitegrid')
print("Librerías cargadas.")

### 2. Carga del conjunto de datos


In [ ]:
df = pd.read_excel('online_shoppers_intention.xlsx')
y = df['Revenue'].astype(int).reset_index(drop=True)  # reservada solo para evaluación

print(f"Dimensiones: {df.shape}")
print(f"Porcentaje de compras (Revenue=True): {y.mean()*100:.1f}%")
df.head()

### 3. Análisis exploratorio de datos (EDA)

Se realiza un análisis exploratorio centrado en los aspectos que van a condicionar las decisiones de preprocesamiento: valores nulos, distribución de variables numéricas (asimetría, outliers, inflación de ceros), correlaciones entre variables, y frecuencia de categorías raras.

#### 3.1. Valores nulos


In [ ]:
print("Valores nulos por columna:")
print(df.isnull().sum().sum(), "valores nulos en total")

El dataset **no contiene valores nulos**, por lo que no es necesario aplicar imputación ni eliminación.

#### 3.2. Variables numéricas: asimetría e inflación de ceros


In [ ]:
numerical_cols = df.select_dtypes(include=['int64','float64']).columns.tolist()
numerical_cols.remove('Revenue') if 'Revenue' in numerical_cols else None

skew_report = df[numerical_cols].skew().sort_values(ascending=False)
zeros_report = (df[numerical_cols] == 0).mean().sort_values(ascending=False) * 100

print("--- Asimetría (skewness) ---")
print(skew_report.round(2))
print("\n--- Porcentaje de ceros por variable ---")
print(zeros_report.round(1))

filas = int(len(numerical_cols)/3) + 1
plt.figure(figsize=(15,10))
for i, col in enumerate(numerical_cols):
    plt.subplot(filas, 3, i+1)
    sns.histplot(df[col], kde=True)
    plt.title(col); plt.xlabel(''); plt.ylabel('')
plt.tight_layout(); plt.show()

**Hallazgos:**
- Todas las variables de duración presentan **asimetría superior a 5**; `PageValues` tiene skew ≈ 6,4.
- **Inflación de ceros** severa: `SpecialDay` 89,9%, `Informational` 78-80%, `PageValues` 77,9%.
- Estas distribuciones justifican aplicar una transformación logarítmica y recortar los valores extremos antes de escalar.

#### 3.3. Correlaciones entre variables numéricas


In [ ]:
corr = df[numerical_cols].corr()
plt.figure(figsize=(10,8))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', center=0, square=True)
plt.title('Matriz de correlación — variables numéricas')
plt.tight_layout(); plt.show()

**Hallazgos:**
- `BounceRates` y `ExitRates` están correlacionadas **r = 0,91** — son prácticamente redundantes.
- Cada columna *count* (`Administrative`, `Informational`, `ProductRelated`) está muy correlacionada (r = 0,60-0,86) con su correspondiente columna *Duration*.
- Esto sugiere eliminar las columnas redundantes para reducir la dimensionalidad sin perder información.

#### 3.4. Variables categóricas


In [ ]:
categorical_cols = ['Month','OperatingSystems','Browser','Region','TrafficType','VisitorType','Weekend']
for col in categorical_cols:
    counts = df[col].value_counts()
    print(f"\n--- {col} ({len(counts)} categorías) ---")
    print(counts.to_string())

**Hallazgos:**
- `VisitorType = 'Other'` aparece solo en **85 filas** — categoría residual.
- Varios códigos de `OperatingSystems`, `Browser` y `TrafficType` tienen **menos de 50 observaciones**, lo que introduciría ruido tras el one-hot encoding.
- Estas categorías raras se agruparán en una etiqueta común `'Other'` antes de codificar.


## Parte III: Preprocesamiento

El preprocesamiento se diseña específicamente para los algoritmos de clustering basados en distancia y densidad, que son sensibles tanto a la escala como a las distribuciones asimétricas. Las decisiones se justifican en los hallazgos del EDA.

| Paso | Acción | Razón |
|---|---|---|
| 1 | Eliminar `Revenue` | Variable objetivo — excluida del clustering |
| 2 | Eliminar `BounceRates` | r = 0,913 con `ExitRates` — prácticamente duplicada |
| 3 | Eliminar `Administrative`, `Informational`, `ProductRelated` | r = 0,60-0,86 con sus `Duration`; las duraciones son más informativas |
| 4 | Binarizar `SpecialDay` → 0/1 | 89,9% de ceros: inútil como continua |
| 5 | Fusionar categorías raras | Códigos con <50 observaciones se reagrupan |
| 6 | `np.log1p` en continuas | Reduce la fuerte asimetría positiva |
| 7 | Recorte al percentil 99 | Elimina los valores extremos residuales |
| 8 | `StandardScaler` | Necesario para métricas de distancia |
| 9 | `OneHotEncoder` en categóricas | Codificación correcta para variables nominales |


In [ ]:
# --- Paso 1: eliminar Revenue (target) ---
data = df.drop(columns=['Revenue']).copy()

# --- Paso 2: eliminar BounceRates (r=0.913 con ExitRates) ---
data = data.drop(columns=['BounceRates'])

# --- Paso 3: eliminar count cols, conservar Duration (más informativas) ---
data = data.drop(columns=['Administrative','Informational','ProductRelated'])

# --- Paso 4: binarizar SpecialDay ---
data['SpecialDay'] = (data['SpecialDay'] > 0).astype(int)

# --- Paso 5: fusionar categorías raras ---
# VisitorType: 'Other' (85 filas) -> 'New_Visitor'
data['VisitorType'] = data['VisitorType'].replace('Other', 'New_Visitor')

# OperatingSystems: {5,6,7,8} -> 'Other'
data['OperatingSystems'] = data['OperatingSystems'].astype(str).replace(
    {'5':'Other','6':'Other','7':'Other','8':'Other'})

# Browser y TrafficType: valores con <50 filas -> 'Other'
for col in ['Browser','TrafficType']:
    counts = df[col].value_counts()
    rare = counts[counts < 50].index.astype(str).tolist()
    data[col] = data[col].astype(str).apply(lambda x: 'Other' if x in rare else x)

data['Region'] = data['Region'].astype(str)

# --- Columnas según su papel ---
continuous_cols = ['Administrative_Duration','Informational_Duration',
                   'ProductRelated_Duration','ExitRates','PageValues']
binary_cols = ['SpecialDay','Weekend']
categorical_cols = ['Month','VisitorType','OperatingSystems','Browser','Region','TrafficType']

# --- Paso 6: log1p en continuas ---
for col in continuous_cols:
    data[col] = np.log1p(data[col])

# --- Paso 7: recorte al p99 ---
for col in continuous_cols:
    p99 = data[col].quantile(0.99)
    data[col] = data[col].clip(upper=p99)

# --- Pasos 8-9: escalado + one-hot encoding ---
data['Weekend'] = data['Weekend'].astype(int)
numeric_to_scale = continuous_cols + binary_cols

preprocessor = ColumnTransformer(transformers=[
    ('num', StandardScaler(), numeric_to_scale),
    ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), categorical_cols),
], remainder='drop')

X = preprocessor.fit_transform(data)
cat_names = preprocessor.named_transformers_['cat'].get_feature_names_out(categorical_cols).tolist()
X_df = pd.DataFrame(X, columns=numeric_to_scale + cat_names)

print(f"Dimensiones finales: {X_df.shape}")
print(f"¿Algún NaN?: {X_df.isnull().any().any()}")
X_df.head()

Tras el preprocesamiento, el conjunto de datos tiene **12.330 filas × 55 columnas**, todas numéricas y estandarizadas. La variable `y` (Revenue) queda reservada únicamente para evaluación posterior.


## Parte IV: K-Means

K-Means es el punto de partida natural en cualquier tarea de clustering: es rápido, escala bien a 12.330 × 55, y su función objetivo (SSE / inertia) es un criterio geométrico directo. Tras estandarizar las variables, la distancia euclídea resulta adecuada. Su limitación principal es que asume clústeres convexos y de tamaño similar.

### 1. Selección del número de clústeres
Se barre `k` de 2 a 10, observando simultáneamente el **método del codo** (SSE) y el **coeficiente de silueta**.


In [ ]:
K_RANGE = range(2, 11)
sse, sil_km = [], []

for k in K_RANGE:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = km.fit_predict(X_df)
    sse.append(km.inertia_)
    s = silhouette_score(X_df, labels, sample_size=3000, random_state=42)
    sil_km.append(s)
    print(f"k={k:2d}  SSE={km.inertia_:>10,.0f}  Silhouette={s:.4f}")

fig, axes = plt.subplots(1, 2, figsize=(12,4))
axes[0].plot(list(K_RANGE), sse, marker='o')
axes[0].set_title('Método del Codo — SSE vs k'); axes[0].set_xlabel('k'); axes[0].set_ylabel('SSE')
axes[1].plot(list(K_RANGE), sil_km, marker='o', color='orange')
axes[1].set_title('Silhouette Score vs k'); axes[1].set_xlabel('k'); axes[1].set_ylabel('Silhouette')
plt.tight_layout(); plt.show()

**Análisis de la selección.** El SSE presenta un codo suave en k=4-5, mientras que la silueta alcanza su **máximo en k=6 (0,1860)**. Se escoge **k=6** porque maximiza la silueta y porque, como se verá a continuación, la distribución de `Revenue` confirma que los clústeres adicionales capturan estructura real (un grupo concentra el 65,9% de compradores, otro es casi puramente no-comprador). El pequeño incremento de SSE que se sacrifica respecto a k=4 se compensa con una segmentación bastante más rica.

### 2. Modelo final y evaluación


In [ ]:
BEST_K_KM = 6
km_final = KMeans(n_clusters=BEST_K_KM, random_state=42, n_init=10)
labels_km = km_final.fit_predict(X_df)

final_sil_km = silhouette_score(X_df, labels_km, sample_size=3000, random_state=42)
final_sse_km = km_final.inertia_
print(f"SSE final:       {final_sse_km:,.0f}")
print(f"Silhouette final: {final_sil_km:.4f}")

def revenue_report(labels, y, name=''):
    res = pd.DataFrame({'cluster': labels, 'Revenue': y})
    rows = []
    ents = []
    for c in sorted(res['cluster'].unique()):
        grp = res[res['cluster']==c]['Revenue']
        p = grp.mean()
        probs = np.array([p, 1-p]); probs = probs[probs>0]
        h = entropy(probs, base=2)
        if c != -1:
            ents.append(h)
        rows.append({'cluster': c, 'n': len(grp),
                     'buyers_%': round(p*100,1), 'entropy': round(h,4)})
    out = pd.DataFrame(rows)
    print(f"\n-- Distribución de Revenue por clúster ({name}) --")
    print(out.to_string(index=False))
    print(f"Entropía media: {np.mean(ents):.4f}")
    return out, np.mean(ents)

report_km, avg_ent_km = revenue_report(labels_km, y, 'K-Means k=6')

**Interpretación del resultado.** K-Means identifica una segmentación coherente con los objetivos del negocio:
- Un clúster concentra el **65,9% de compradores** (alto interés de compra).
- Un clúster prácticamente puro de **no-compradores (0,8%)**.
- Un grupo grande de baja intención (~5 000 usuarios, 4,7% compradores).
- Tres grupos intermedios con intención mixta.

### 3. Visualización con PCA


In [ ]:
pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_df)
pca_df = pd.DataFrame(X_pca, columns=['PC1','PC2'])
pca_df['Cluster'] = labels_km
pca_df['Revenue'] = y.values

plt.figure(figsize=(10,7))
sns.scatterplot(x='PC1', y='PC2', hue='Cluster', style='Revenue',
                data=pca_df, palette='viridis', alpha=0.6, s=40)
plt.title(f'K-Means (k={BEST_K_KM}) — Proyección PCA')
plt.tight_layout(); plt.show()

## Parte V: Clustering Jerárquico (Agglomerative)

Como **alternativa determinista y no paramétrica** a K-Means se utiliza clustering aglomerativo con enlace **Ward**. Ward es el único enlace que minimiza directamente la varianza intra-clúster en cada fusión — el mismo criterio que SSE en K-Means — lo que hace ambos métodos directamente comparables. Además, no depende de semilla aleatoria, por lo que los resultados son totalmente reproducibles, y ofrece el dendrograma como verificación visual de la estructura jerárquica.

### 1. Barrido de k con silueta


In [ ]:
sil_agg = []
for k in K_RANGE:
    agg = AgglomerativeClustering(n_clusters=k, linkage='ward')
    labels = agg.fit_predict(X_df)
    s = silhouette_score(X_df, labels, sample_size=3000, random_state=42)
    sil_agg.append(s)
    print(f"k={k:2d}  Silhouette={s:.4f}")

plt.figure(figsize=(7,4))
plt.plot(list(K_RANGE), sil_agg, marker='o', color='steelblue')
plt.title('Agglomerative (Ward) — Silhouette vs k')
plt.xlabel('k'); plt.ylabel('Silhouette')
plt.tight_layout(); plt.show()

### 2. Dendrograma (submuestra)

El dendrograma se construye sobre una submuestra de 500 filas para que sea legible; se muestran las **20 últimas fusiones**, suficientes para identificar los cortes naturales de la jerarquía.


In [ ]:
np.random.seed(42)
sample_idx = np.random.choice(len(X_df), size=500, replace=False)
Z = linkage(X_df.iloc[sample_idx].values, method='ward')

plt.figure(figsize=(14,5))
dendrogram(Z, truncate_mode='lastp', p=20, leaf_rotation=90, leaf_font_size=10)
plt.title('Dendrograma (Ward, submuestra 500 filas, últimas 20 fusiones)')
plt.xlabel('Clúster'); plt.ylabel('Distancia')
plt.tight_layout(); plt.show()

**Selección.** La silueta alcanza su máximo en **k=5 (0,1874)**. El dendrograma muestra un salto notable antes de la fusión final en torno a 5 grupos, lo que refuerza la decisión.

### 3. Modelo final y evaluación


In [ ]:
BEST_K_AGG = 5
agg_final = AgglomerativeClustering(n_clusters=BEST_K_AGG, linkage='ward')
labels_agg = agg_final.fit_predict(X_df)

final_sil_agg = silhouette_score(X_df, labels_agg, sample_size=3000, random_state=42)
print(f"Silhouette final: {final_sil_agg:.4f}")

report_agg, avg_ent_agg = revenue_report(labels_agg, y, 'Agglomerative k=5')

**Interpretación.** Agglomerative recupera una estructura casi idéntica a la de K-Means: un gran clúster de baja intención (~6 800 usuarios), un grupo de alta intención con **65,5% de compradores** y otro casi puro de no-compradores (0,4%). Esta coincidencia entre dos métodos independientes refuerza la validez de la segmentación.


## Parte VI: DBSCAN

DBSCAN se incluye como **algoritmo basado en densidad**. Es útil en este contexto por dos razones: (1) puede encontrar clústeres de forma arbitraria separados por regiones de baja densidad — por ejemplo, una pequeña cohorte de sesiones de alto valor geométricamente aislada —, y (2) identifica puntos de ruido, lo que resulta interesante para detección de anomalías.

### 1. Selección de hiperparámetros

`min_samples=10` se fija como regla de referencia (aproximadamente `2·d` sería demasiado alto en 55 dimensiones; 10 evita sobre-fragmentación). Para `eps` se combinan dos técnicas:

1. **Gráfica de k-distancias**: se ordena de mayor a menor la distancia al k-ésimo vecino más cercano (k=9) y se busca el codo.
2. **Barrido de `eps`** evaluando número de clústeres, porcentaje de ruido y silueta.


In [ ]:
MIN_SAMPLES = 10
k = MIN_SAMPLES - 1
nbrs = NearestNeighbors(n_neighbors=k).fit(X_df)
distances, _ = nbrs.kneighbors(X_df)
k_distances = np.sort(distances[:, -1])[::-1]

plt.figure(figsize=(8,4))
plt.plot(k_distances)
plt.title(f'Gráfica k-distancias (k={k}) — el codo sugiere eps')
plt.xlabel('Puntos ordenados'); plt.ylabel(f'Distancia al {k}-NN')
plt.tight_layout(); plt.show()

In [ ]:
import math
EPS_VALUES = [1.5, 2.0, 2.5, 3.0, 3.5, 4.0, 5.0]
sweep = []
print(f"{'eps':>5}  {'clusters':>8}  {'noise%':>7}  {'silhouette':>10}")
for eps in EPS_VALUES:
    db = DBSCAN(eps=eps, min_samples=MIN_SAMPLES)
    labels = db.fit_predict(X_df)
    n_cl = len(set(labels)) - (1 if -1 in labels else 0)
    n_noise = (labels==-1).sum()
    npct = n_noise/len(labels)*100
    if n_cl >= 2:
        mask = labels != -1
        s = silhouette_score(X_df[mask], labels[mask], sample_size=3000, random_state=42)
    else:
        s = float('nan')
    sweep.append((eps, n_cl, n_noise, npct, s))
    print(f"{eps:>5.1f}  {n_cl:>8d}  {npct:>6.1f}%  {s:>10.4f}")

**Selección.** Con `eps=1.5` o `2.0` el algoritmo fragmenta en exceso y genera mucho ruido (52% y 18%). A partir de `eps≥3.5` colapsa a un único clúster. **`eps=2.5`** ofrece el mejor equilibrio: silueta máxima (0,2110) con solo 0,7% de ruido.

### 2. Modelo final y evaluación


In [ ]:
BEST_EPS = 2.5
db_final = DBSCAN(eps=BEST_EPS, min_samples=MIN_SAMPLES)
labels_db = db_final.fit_predict(X_df)

n_cl_db = len(set(labels_db)) - (1 if -1 in labels_db else 0)
n_noise_db = (labels_db==-1).sum()
mask = labels_db != -1
final_sil_db = silhouette_score(X_df[mask], labels_db[mask], sample_size=3000, random_state=42)

print(f"Clústeres: {n_cl_db}")
print(f"Ruido:     {n_noise_db} ({n_noise_db/len(labels_db)*100:.1f}%)")
print(f"Silhouette (sin ruido): {final_sil_db:.4f}")

report_db, avg_ent_db = revenue_report(labels_db, y, 'DBSCAN eps=2.5')

**Limitación.** DBSCAN colapsa a **dos clústeres** muy heterogéneos, consecuencia directa de la **maldición de la dimensionalidad**: en 55 dimensiones las distancias entre puntos tienden a uniformizarse y es difícil encontrar regiones claramente más densas que su entorno. La silueta (0,2110) es la más alta de todos los algoritmos, pero es en parte un artefacto de calcularse solo sobre los puntos no-ruido y de disponer de dos grandes masas bien separadas. Como segmentación práctica resulta demasiado grueso.


## Parte VII: Gaussian Mixture Models (GMM)

GMM es la **contrapartida probabilística** de K-Means: en lugar de asignación dura, ajusta una mezcla de gaussianas y cada punto recibe una probabilidad de pertenecer a cada componente. Es apropiado cuando los clústeres solapan o tienen orientaciones/tamaños distintos. Se emplea covarianza `full` (cada componente con su propia matriz arbitraria) porque las alternativas `diag` o `spherical` impondrían restricciones de simetría poco realistas en un espacio conductual de 55 dimensiones. La selección de `k` se guía por **BIC**, que penaliza la complejidad frente al ajuste.

### 1. Barrido de k (BIC, AIC y silueta)


In [ ]:
bic, aic, sil_gmm = [], [], []
print(f"{'k':>3}  {'BIC':>14}  {'AIC':>14}  {'Silhouette':>10}")
for k in K_RANGE:
    gmm = GaussianMixture(n_components=k, covariance_type='full',
                          random_state=42, max_iter=200)
    gmm.fit(X_df)
    labels = gmm.predict(X_df)
    b, a = gmm.bic(X_df), gmm.aic(X_df)
    s = silhouette_score(X_df, labels, sample_size=3000, random_state=42)
    bic.append(b); aic.append(a); sil_gmm.append(s)
    print(f"{k:>3}  {b:>14,.0f}  {a:>14,.0f}  {s:>10.4f}")

fig, axes = plt.subplots(1, 2, figsize=(12,4))
axes[0].plot(list(K_RANGE), bic, marker='o', label='BIC')
axes[0].plot(list(K_RANGE), aic, marker='s', label='AIC')
axes[0].set_title('GMM — BIC y AIC vs k'); axes[0].set_xlabel('k'); axes[0].legend()
axes[1].plot(list(K_RANGE), sil_gmm, marker='o', color='green')
axes[1].set_title('GMM — Silhouette vs k'); axes[1].set_xlabel('k')
plt.tight_layout(); plt.show()

**Selección.** BIC decrece de forma **monótona** hasta k=10, sin plateau. Este comportamiento es característico de espacios de alta dimensión con covarianza `full`, donde cada componente añadido aporta muchos grados de libertad. Se elige **k=10** (mínimo BIC). La silueta es baja para todos los valores de k, lo cual es esperable: GMM particiona densidad probabilística, no cohesión geométrica.

### 2. Modelo final y evaluación


In [ ]:
BEST_K_GMM = 10
gmm_final = GaussianMixture(n_components=BEST_K_GMM, covariance_type='full',
                            random_state=42, max_iter=200)
gmm_final.fit(X_df)
labels_gmm = gmm_final.predict(X_df)

final_sil_gmm = silhouette_score(X_df, labels_gmm, sample_size=3000, random_state=42)
print(f"BIC:        {gmm_final.bic(X_df):,.0f}")
print(f"AIC:        {gmm_final.aic(X_df):,.0f}")
print(f"Silhouette: {final_sil_gmm:.4f}")

report_gmm, avg_ent_gmm = revenue_report(labels_gmm, y, 'GMM k=10')

**Interpretación.** GMM descubre más variación en la composición de `Revenue`: tres clústeres son casi puros no-compradores (entropía 0,04–0,17) y varios componentes intermedios muestran tasas de compra del 30-42%. Esta granularidad tiene un coste: la silueta cae a 0,022, indicando solapamiento fuerte entre componentes. El modelo es estadísticamente el mejor ajustado, pero menos interpretable como segmentación operativa.


## Parte VIII: Comparación de resultados

### Métricas resumen


In [ ]:
summary = pd.DataFrame([
    {'Algoritmo':'K-Means',        'k':BEST_K_KM,  'Silhouette':round(final_sil_km,4),
     'SSE':round(final_sse_km,0),  'Entropía media':round(avg_ent_km,4)},
    {'Algoritmo':'Agglomerative',  'k':BEST_K_AGG, 'Silhouette':round(final_sil_agg,4),
     'SSE':None,                   'Entropía media':round(avg_ent_agg,4)},
    {'Algoritmo':'DBSCAN',         'k':n_cl_db,    'Silhouette':round(final_sil_db,4),
     'SSE':None,                   'Entropía media':round(avg_ent_db,4)},
    {'Algoritmo':'GMM',            'k':BEST_K_GMM, 'Silhouette':round(final_sil_gmm,4),
     'SSE':None,                   'Entropía media':round(avg_ent_gmm,4)},
])
summary

### Análisis comparativo

**Estructura de los clústeres.** K-Means (k=6) y Agglomerative (k=5) convergen en una fotografía muy similar de los datos: un gran grupo de baja intención (~55-84% de las sesiones), un grupo concentrado de alta intención con ~65% de compradores, un grupo casi puro de no-compradores (<1%) y varios grupos intermedios. Esta **coincidencia entre dos métodos independientes** — uno con inicialización aleatoria y esférico, el otro determinista y basado en varianza — aporta mucha confianza en que la segmentación refleja estructura real en los datos.

**Limitación de DBSCAN.** El colapso a dos clústeres es consecuencia directa de la maldición de la dimensionalidad: en 55 dimensiones las distancias se uniformizan y resulta muy difícil detectar variaciones de densidad locales. El 0,7% de ruido es sano pero los dos clústeres (n=11 030 y n=1 217) son demasiado gruesos para segmentación accionable. Aunque su silueta (0,2110) es la más alta, se calcula solo sobre puntos no-ruido y refleja dos grandes masas bien separadas, no estructura fina. DBSCAN sería más útil aplicado sobre una proyección PCA 2D-3D o sobre un subconjunto puramente continuo.

**Compromiso de GMM.** GMM con k=10 captura la mayor variación en la composición de `Revenue` y alcanza el mejor ajuste estadístico (BIC mínimo), pero a costa de una silueta muy baja: los componentes se solapan fuertemente en el espacio. Es el modelo adecuado si se necesitan **probabilidades de pertenencia** para tareas posteriores, pero resulta menos directo para segmentación operativa.

**Entropía de Revenue.** K-Means y Agglomerative alcanzan una entropía media muy similar (~0,478), lo que significa que sus clústeres son igual de puros respecto a la variable objetivo. La entropía algo más alta de GMM (0,556) refleja sus componentes solapados. Menor entropía = clústeres más puros = segmentos más útiles para marketing dirigido.


## Parte IX: Conclusiones

1. **Algoritmo recomendado: Agglomerative (Ward, k=5).** Obtiene la mejor silueta entre los métodos con segmentación fina (0,1874 frente a 0,1860 de K-Means), es **totalmente determinista**, y produce una división por `Revenue` igual de interpretable que K-Means. Es la opción más defendible como resultado final de la práctica.

2. **K-Means (k=6)** queda como segunda opción muy cercana y resulta preferible cuando interesa la **escalabilidad** o la posibilidad de refitting rápido (por ejemplo, datos en streaming). La consistencia con Agglomerative es una validación cruzada fuerte de la segmentación.

3. **GMM (k=10)** es la opción adecuada si se necesitan **probabilidades de pertenencia** (por ejemplo, para scoring suave o para alimentar un modelo downstream), aceptando a cambio clústeres menos geométricamente compactos.

4. **DBSCAN** no produce segmentación útil en este espacio de 55 dimensiones. Para sacarle partido habría que aplicarlo tras una fuerte reducción de dimensionalidad, o restringido a las variables continuas de comportamiento.

5. **Hallazgo de negocio.** Los tres métodos (K-Means, Agglomerative y GMM) coinciden en identificar un **segmento claro de alta intención de compra** (~12-15% de las sesiones con 42-66% de conversión) y un **segmento grande de baja intención**. Esto demuestra que existe estructura real en los datos y que la segmentación por comportamiento es viable como punto de partida para acciones de marketing dirigidas.

6. **Papel del preprocesamiento.** La transformación logarítmica, el recorte al p99 y la eliminación de variables redundantes han sido clave para que los métodos basados en distancia encuentren estructura. Sin ellas, las variables de duración (con skew > 5) habrían dominado las distancias y degradado los resultados.
